In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP24A - CELL 1
# Environment + paths + load FROZEN Step23 objects
# ============================================================

from pathlib import Path

import sys
import json
import warnings

import numpy as np
import pandas as pd

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy import sparse

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# 1. Paths
# ============================================================

STEP23_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

STEP24_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step24_Bulk_to_scRNA_Projection')
)


DATA_DIR = STEP24_ROOT / "Data"
RESULTS_DIR = STEP24_ROOT / "Results"
FIGURES_DIR = STEP24_ROOT / "Figures"
LOGS_DIR = STEP24_ROOT / "Logs"
REFERENCE_DIR = STEP24_ROOT / "Reference"


for folder in [
    DATA_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    LOGS_DIR,
    REFERENCE_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# 2. Input files from FROZEN Step23
# ============================================================

FINAL_STATE_INPUT = (
    STEP23_ROOT
    / "Data"
    / "GSE149614_STEP23_FINAL_MALIGNANT_STATES.h5ad"
)

FINAL_PB_INPUT = (
    STEP23_ROOT
    / "Data"
    / "GSE149614_STEP23_FINAL_State_Pseudobulk_COUNTS.h5ad"
)

FINAL_MAPPING_INPUT = (
    STEP23_ROOT
    / "Results"
    / "Step23_Final_MalignantState_Mapping.csv"
)

FINAL_SUPPORT_INPUT = (
    STEP23_ROOT
    / "Results"
    / "Step23_Final_State_PatientSupport.csv"
)


for path in [
    FINAL_STATE_INPUT,
    FINAL_PB_INPUT,
    FINAL_MAPPING_INPUT,
    FINAL_SUPPORT_INPUT
]:

    assert path.exists(), f"Missing file:\n{path}"


# ============================================================
# 3. Load
# ============================================================

adata = sc.read_h5ad(
    FINAL_STATE_INPUT
)

adata_pb_counts = sc.read_h5ad(
    FINAL_PB_INPUT
)

final_mapping = pd.read_csv(
    FINAL_MAPPING_INPUT
)

final_support = pd.read_csv(
    FINAL_SUPPORT_INPUT
)


STATE_KEY = "final_malignant_state_id"

STATE_ORDER = [
    f"MS{i}"
    for i in range(1, 9)
]


print("=" * 115)
print("STEP24A - FROZEN BULK-TO-SINGLE-CELL PROJECTION")
print("=" * 115)

print("\nPython:")
print(sys.executable)

print("\nScanpy:")
print(sc.__version__)

print("\nSingle-cell object:")
print(adata.shape)

print("\nFinal-state pseudobulk object:")
print(adata_pb_counts.shape)

print("\nFinal malignant states:")

display(
    adata.obs[
        STATE_KEY
    ]
    .astype(str)
    .value_counts()
    .reindex(
        STATE_ORDER
    )
    .to_frame(
        "N_cells"
    )
)

print("\nImportant design principle:")
print(
    "TCGA-LIHC bulk and GSE149614 scRNA are independent cohorts."
)
print(
    "Step24 performs cross-cohort biological projection, "
    "not direct patient-level RiskScore correlation."
)

In [ ]:
# ============================================================
# STEP24A - CELL 2
# Publication plotting settings
# ============================================================

plt.rcParams["font.sans-serif"] = [
    "Arial",
    "DejaVu Sans"
]

plt.rcParams["axes.unicode_minus"] = False


mpl.rcParams.update({

    "figure.dpi": 120,
    "savefig.dpi": 600,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 11,

    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,

    "axes.linewidth": 0.8,
    "axes.grid": False,

    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


NPG_COLORS = [

    "#E64B35",
    "#4DBBD5",
    "#00A087",
    "#3C5488",
    "#F39B7F",
    "#8491B4",
    "#91D1C2",
    "#7E6148"

]


def save_publication_figure(
    fig,
    filename
):

    png_path = (
        FIGURES_DIR
        / f"{filename}.png"
    )

    pdf_path = (
        FIGURES_DIR
        / f"{filename}.pdf"
    )


    # Save BEFORE show
    fig.savefig(
        png_path,
        dpi=600,
        bbox_inches="tight",
        facecolor="white"
    )

    fig.savefig(
        pdf_path,
        bbox_inches="tight",
        facecolor="white"
    )


    print(
        "Saved:",
        png_path.name
    )

    print(
        "Saved:",
        pdf_path.name
    )

In [ ]:
# ============================================================
# STEP24A - CELL 3
# Define FROZEN bulk anchors
# ============================================================

ORIGINAL_FINAL_TOP10 = [

    "HOXD9",
    "RFESD",
    "RPSAP53",
    "TRGC1",
    "FAM227B",
    "AC015871.6",
    "LINC00324",
    "ACOXL",
    "PTMAP4",
    "ANKRD20A7P"

]


STABLE_GENES10 = [

    "HOXD9",
    "RPSAP53",
    "RFESD",
    "LINC00324",
    "TRGC1",
    "ANKRD20A7P",
    "FAM227B",
    "BOP1",
    "GLYATL1",
    "PRR11"

]


HUB18_BLUE = [

    "CENPA",
    "KIF2C",
    "CCNB1",
    "CDC20",
    "PRR11",
    "CDCA8"

]


HUB18_RED = [

    "SLC52A2",
    "RPL8",
    "BOP1",
    "ZFAS1",
    "SNHG6",
    "RPL7P9"

]


HUB18_BROWN = [

    "CYP8B1",
    "GLYATL1",
    "PLG",
    "ACSM2A",
    "ABAT",
    "MAT1A"

]


STABLEBIO7 = [

    "mTORC1 Signaling",
    "Checkpoint_TNFRSF4",
    "Cell_B_cells",
    "Function_HLA_activity",
    "Checkpoint_CD276",
    "Pancreas Beta Cells",
    "Spermatogenesis"

]


BULK_HIGH_RISK_DIRECTION = {

    "Hub18_Blue":
        "Higher_in_bulk_high_risk",

    "Hub18_Red":
        "Higher_in_bulk_high_risk",

    "Hub18_Brown":
        "Lower_in_bulk_high_risk"

}


anchor_definitions = {

    "Original Final Top10":
        ORIGINAL_FINAL_TOP10,

    "StableGenes10":
        STABLE_GENES10,

    "Hub18_Blue":
        HUB18_BLUE,

    "Hub18_Red":
        HUB18_RED,

    "Hub18_Brown":
        HUB18_BROWN

}


anchor_json = {

    "Original_Final_Top10":
        ORIGINAL_FINAL_TOP10,

    "StableGenes10":
        STABLE_GENES10,

    "Hub18_Blue":
        HUB18_BLUE,

    "Hub18_Red":
        HUB18_RED,

    "Hub18_Brown":
        HUB18_BROWN,

    "StableBio7":
        STABLEBIO7,

    "Hub18_bulk_high_risk_direction":
        BULK_HIGH_RISK_DIRECTION,

    "Important":
        (
            "Original Final Top10 and StableGenes10 are different "
            "frozen feature sets and must not be conflated."
        )
}


ANCHOR_JSON_PATH = (
    REFERENCE_DIR
    / "Step24A_Frozen_BulkAnchor_Definitions.json"
)


with open(
    ANCHOR_JSON_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        anchor_json,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    "Frozen anchor definitions saved:"
)

print(
    ANCHOR_JSON_PATH
)

In [ ]:
# ============================================================
# STEP24A - CELL 4
# Bulk-anchor coverage audit in scRNA
# ============================================================

coverage_rows = []


for anchor_name, genes in (
    anchor_definitions.items()
):

    detected = [

        gene
        for gene in genes
        if gene in adata.var_names
    ]


    missing = [

        gene
        for gene in genes
        if gene not in adata.var_names
    ]


    coverage_rows.append({

        "Anchor":
            anchor_name,

        "N_defined":
            len(genes),

        "N_detected":
            len(detected),

        "Coverage_fraction":
            len(detected)
            /
            len(genes),

        "Detected_genes":
            "|".join(
                detected
            ),

        "Missing_genes":
            "|".join(
                missing
            )
    })


coverage_table = pd.DataFrame(
    coverage_rows
)


COVERAGE_PATH = (
    RESULTS_DIR
    / "Step24A_BulkAnchor_scRNA_Coverage.csv"
)


coverage_table.to_csv(
    COVERAGE_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 100)
print("BULK-ANCHOR COVERAGE IN scRNA")
print("=" * 100)


display(

    coverage_table[
        [
            "Anchor",
            "N_defined",
            "N_detected",
            "Coverage_fraction",
            "Detected_genes",
            "Missing_genes"
        ]
    ]

    .round(3)
)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP24A - CELL 5
# Locate Step17 frozen single-cell anchor reference
# ============================================================

STEP17_ROOT = Path(
    _hcc_path('LIHC_RNA_Step17_Bulk_Freeze')
)


reference_candidates = []


if STEP17_ROOT.exists():

    reference_candidates = list(

        STEP17_ROOT.rglob(
            "SingleCell_Anchor_Features.tsv"
        )
    )


print(
    "SingleCell_Anchor_Features.tsv found:",
    len(reference_candidates)
)


for path in reference_candidates:

    print(
        path
    )


if len(reference_candidates) == 1:

    bulk_anchor_reference = pd.read_csv(
        reference_candidates[0],
        sep="\t"
    )


    snapshot_path = (
        REFERENCE_DIR
        / "Step17_SingleCell_Anchor_Features_snapshot.csv"
    )


    bulk_anchor_reference.to_csv(
        snapshot_path,
        index=False,
        encoding="utf-8-sig"
    )


    print(
        "\nReference snapshot saved."
    )


    print(
        "Shape:",
        bulk_anchor_reference.shape
    )


    display(
        bulk_anchor_reference.head(20)
    )


else:

    bulk_anchor_reference = None


    print(
        "\nNo unique Step17 reference file resolved."
    )

    print(
        "Continue using the locked definitions from Cell 3."
    )

In [ ]:
# ============================================================
# STEP24A - CELL 6
# StableBio7 mapping audit
# ============================================================

stablebio_mapping_rows = [

    {
        "StableBio_feature":
            "mTORC1 Signaling",

        "Direct_scRNA_mapping":
            False,

        "Mapping":
            "Requires frozen pathway gene-set definition",

        "Status":
            "Deferred"
    },

    {
        "StableBio_feature":
            "Checkpoint_TNFRSF4",

        "Direct_scRNA_mapping":
            "TNFRSF4" in adata.var_names,

        "Mapping":
            "TNFRSF4",

        "Status":
            (
                "Direct_gene"
                if
                "TNFRSF4" in adata.var_names
                else
                "Gene_missing"
            )
    },

    {
        "StableBio_feature":
            "Cell_B_cells",

        "Direct_scRNA_mapping":
            False,

        "Mapping":
            (
                "Bulk immune-cell abundance feature; "
                "not a malignant-cell intrinsic feature"
            ),

        "Status":
            "Not_directly_projected"
    },

    {
        "StableBio_feature":
            "Function_HLA_activity",

        "Direct_scRNA_mapping":
            False,

        "Mapping":
            "Requires frozen HLA-activity gene-set definition",

        "Status":
            "Deferred"
    },

    {
        "StableBio_feature":
            "Checkpoint_CD276",

        "Direct_scRNA_mapping":
            "CD276" in adata.var_names,

        "Mapping":
            "CD276",

        "Status":
            (
                "Direct_gene"
                if
                "CD276" in adata.var_names
                else
                "Gene_missing"
            )
    },

    {
        "StableBio_feature":
            "Pancreas Beta Cells",

        "Direct_scRNA_mapping":
            False,

        "Mapping":
            "Requires frozen pathway gene-set definition",

        "Status":
            "Deferred"
    },

    {
        "StableBio_feature":
            "Spermatogenesis",

        "Direct_scRNA_mapping":
            False,

        "Mapping":
            "Requires frozen pathway gene-set definition",

        "Status":
            "Deferred"
    }
]


stablebio_mapping = pd.DataFrame(
    stablebio_mapping_rows
)


STABLEBIO_PATH = (
    RESULTS_DIR
    / "Step24A_StableBio7_MappingAudit.csv"
)


stablebio_mapping.to_csv(
    STABLEBIO_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    stablebio_mapping
)

In [ ]:
# ============================================================
# STEP24A - CELL 7
# Final-state pseudobulk logCPM normalization
# ============================================================

adata_pb_log = (
    adata_pb_counts.copy()
)


sc.pp.normalize_total(
    adata_pb_log,
    target_sum=1e6
)


sc.pp.log1p(
    adata_pb_log
)


PB_LOG_PATH = (
    DATA_DIR
    / "GSE149614_STEP24A_FinalState_Pseudobulk_LogCPM.h5ad"
)


adata_pb_log.write_h5ad(
    PB_LOG_PATH,
    compression="gzip"
)


print(
    "Pseudobulk units:",
    adata_pb_log.n_obs
)

print(
    "Genes:",
    adata_pb_log.n_vars
)


display(
    adata_pb_log.obs
    .sort_values(
        [
            "Final_state_id",
            "patient"
        ]
    )
)


print(
    "\nSaved:",
    PB_LOG_PATH
)

In [ ]:
# ============================================================
# STEP24A - CELL 8
# Within-patient centered pseudobulk expression
# ============================================================

if sparse.issparse(
    adata_pb_log.X
):

    X_pb = (
        adata_pb_log.X
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    X_pb = np.asarray(
        adata_pb_log.X,
        dtype=np.float32
    )


centered_matrix = np.full(

    X_pb.shape,

    np.nan,

    dtype=np.float32
)


patients = (

    adata_pb_log.obs[
        "patient"
    ]
    .astype(str)
    .unique()
)


for patient in patients:

    idx = np.where(

        adata_pb_log.obs[
            "patient"
        ]
        .astype(str)
        .values

        ==

        patient

    )[0]


    patient_mean = np.mean(
        X_pb[
            idx,
            :
        ],
        axis=0
    )


    centered_matrix[
        idx,
        :
    ] = (

        X_pb[
            idx,
            :
        ]

        -

        patient_mean
    )


adata_pb_centered = ad.AnnData(

    X=
        centered_matrix,

    obs=
        adata_pb_log.obs.copy(),

    var=
        adata_pb_log.var.copy()
)


CENTERED_PATH = (

    DATA_DIR
    / "GSE149614_STEP24A_FinalState_WithinPatientCentered.h5ad"
)


adata_pb_centered.write_h5ad(
    CENTERED_PATH,
    compression="gzip"
)


print(
    "Saved:",
    CENTERED_PATH
)

In [ ]:
# ============================================================
# STEP24A - CELL 9
# Eligible patient-state units
# ============================================================

MIN_CELLS_PER_PATIENT_STATE = 20


eligible_mask = (

    adata_pb_centered.obs[
        "N_cells"
    ]
    >=
    MIN_CELLS_PER_PATIENT_STATE
)


eligible_obs = (

    adata_pb_centered.obs.loc[
        eligible_mask
    ]
    .copy()
)


print("=" * 105)
print("ELIGIBLE PATIENT-STATE UNITS")
print("=" * 105)


print(
    "Eligible units:",
    len(
        eligible_obs
    )
)


display(

    eligible_obs

    .sort_values(
        [
            "Final_state_id",
            "patient"
        ]
    )
)

In [ ]:
# ============================================================
# STEP24A - CELL 10
# Gene-level projection function
# ============================================================

gene_to_idx = {

    gene: i

    for i, gene
    in enumerate(
        adata_pb_centered.var_names
    )
}


def build_state_gene_projection(
    gene_list,
    anchor_name
):

    detected = [

        gene
        for gene in gene_list
        if gene in gene_to_idx
    ]


    rows = []


    for state in STATE_ORDER:

        state_mask = (

            eligible_mask.values

            &

            (
                adata_pb_centered.obs[
                    "Final_state_id"
                ]
                .astype(str)
                .values
                ==
                state
            )
        )


        state_idx = np.where(
            state_mask
        )[0]


        patients_state = (

            adata_pb_centered.obs.iloc[
                state_idx
            ][
                "patient"
            ]
            .astype(str)
            .tolist()
        )


        for gene in detected:

            gene_idx = (
                gene_to_idx[
                    gene
                ]
            )


            values = (

                centered_matrix[
                    state_idx,
                    gene_idx
                ]
            )


            values = values[
                np.isfinite(
                    values
                )
            ]


            rows.append({

                "Anchor":
                    anchor_name,

                "Final_state_id":
                    state,

                "Gene":
                    gene,

                "N_patients":
                    len(values),

                "Patients":
                    "|".join(
                        patients_state
                    ),

                "Mean_withinPatient_delta":
                    (
                        float(
                            np.mean(values)
                        )
                        if
                        len(values) > 0
                        else
                        np.nan
                    ),

                "Median_withinPatient_delta":
                    (
                        float(
                            np.median(values)
                        )
                        if
                        len(values) > 0
                        else
                        np.nan
                    ),

                "Positive_fraction":
                    (
                        float(
                            np.mean(
                                values > 0
                            )
                        )
                        if
                        len(values) > 0
                        else
                        np.nan
                    )
            })


    return pd.DataFrame(
        rows
    )

In [ ]:
# ============================================================
# STEP24A - CELL 11
# Project Original Final Top10 and StableGenes10
# ============================================================

original_top10_projection = (
    build_state_gene_projection(

        ORIGINAL_FINAL_TOP10,

        "Original Final Top10"
    )
)


stablegenes10_projection = (
    build_state_gene_projection(

        STABLE_GENES10,

        "StableGenes10"
    )
)


ORIGINAL_PROJECTION_PATH = (

    RESULTS_DIR
    / "Step24A_OriginalFinalTop10_GeneProjection.csv"
)


STABLE_PROJECTION_PATH = (

    RESULTS_DIR
    / "Step24A_StableGenes10_GeneProjection.csv"
)


original_top10_projection.to_csv(
    ORIGINAL_PROJECTION_PATH,
    index=False,
    encoding="utf-8-sig"
)


stablegenes10_projection.to_csv(
    STABLE_PROJECTION_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Original Final Top10 projection:"
)

display(
    original_top10_projection
    .round(3)
)


print(
    "\nStableGenes10 projection:"
)

display(
    stablegenes10_projection
    .round(3)
)

In [ ]:
# ============================================================
# STEP24A - CELL 12
# Original Final Top10 projection heatmap
# ============================================================

original_heat = (

    original_top10_projection

    .pivot(
        index=
            "Final_state_id",

        columns=
            "Gene",

        values=
            "Mean_withinPatient_delta"
    )

    .reindex(
        STATE_ORDER
    )
)


original_gene_order = [

    gene
    for gene in ORIGINAL_FINAL_TOP10
    if gene in original_heat.columns
]


original_heat = (
    original_heat[
        original_gene_order
    ]
)


fig, ax = plt.subplots(
    figsize=(8.7, 6.2)
)


values = original_heat.values


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        original_heat.shape[1]
    )
)


ax.set_xticklabels(
    original_heat.columns,
    rotation=45,
    ha="right"
)


ax.set_yticks(
    np.arange(
        len(STATE_ORDER)
    )
)


ax.set_yticklabels(
    STATE_ORDER
)


ax.set_xlabel(
    "Detected Gene"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "Original Final Top10 Projection (Detected Genes Only)"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Mean Within-patient Expression Delta"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step24A_01_OriginalFinalTop10_GeneProjection"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24A - CELL 13
# StableGenes10 projection heatmap
# ============================================================

stable_heat = (

    stablegenes10_projection

    .pivot(
        index=
            "Final_state_id",

        columns=
            "Gene",

        values=
            "Mean_withinPatient_delta"
    )

    .reindex(
        STATE_ORDER
    )
)


stable_gene_order = [

    gene
    for gene in STABLE_GENES10
    if gene in stable_heat.columns
]


stable_heat = (
    stable_heat[
        stable_gene_order
    ]
)


fig, ax = plt.subplots(
    figsize=(9.5, 6.2)
)


values = stable_heat.values


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        stable_heat.shape[1]
    )
)


ax.set_xticklabels(
    stable_heat.columns,
    rotation=45,
    ha="right"
)


ax.set_yticks(
    np.arange(
        len(STATE_ORDER)
    )
)


ax.set_yticklabels(
    STATE_ORDER
)


ax.set_xlabel(
    "Detected Gene"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "StableGenes10 Projection (Detected Genes Only)"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Mean Within-patient Expression Delta"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step24A_02_StableGenes10_GeneProjection"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24A - CELL 14
# Hub18 directional bulk-risk concordance
# ============================================================

blue_detected = [

    gene
    for gene in HUB18_BLUE
    if gene in gene_to_idx
]


red_detected = [

    gene
    for gene in HUB18_RED
    if gene in gene_to_idx
]


brown_detected = [

    gene
    for gene in HUB18_BROWN
    if gene in gene_to_idx
]


print(
    "Blue detected:",
    blue_detected
)

print(
    "\nRed detected:",
    red_detected
)

print(
    "\nBrown detected:",
    brown_detected
)


hub_genes = list(
    dict.fromkeys(

        blue_detected
        +
        red_detected
        +
        brown_detected
    )
)


eligible_indices = np.where(
    eligible_mask.values
)[0]


hub_idx = [

    gene_to_idx[
        gene
    ]

    for gene in hub_genes
]


hub_matrix = (

    centered_matrix[
        eligible_indices,
        :
    ][
        :,
        hub_idx
    ]
    .astype(
        np.float64
    )
)


# ============================================================
# Per-gene standardization across eligible patient-state units
# ============================================================

gene_mean = np.nanmean(
    hub_matrix,
    axis=0
)


gene_sd = np.nanstd(
    hub_matrix,
    axis=0,
    ddof=1
)


gene_sd[
    (~np.isfinite(gene_sd))
    |
    (gene_sd == 0)
] = 1.0


hub_z = (

    hub_matrix
    -
    gene_mean

) / gene_sd


hub_gene_pos = {

    gene: i

    for i, gene
    in enumerate(
        hub_genes
    )
}


def module_mean(
    genes
):

    positions = [

        hub_gene_pos[
            gene
        ]

        for gene in genes
        if gene in hub_gene_pos
    ]


    if len(positions) == 0:

        return np.full(
            len(
                eligible_indices
            ),
            np.nan
        )


    return np.nanmean(

        hub_z[
            :,
            positions
        ],

        axis=1
    )


blue_score = module_mean(
    blue_detected
)


red_score = module_mean(
    red_detected
)


brown_score = module_mean(
    brown_detected
)


hub18_concordance = (

    (
        blue_score
        +
        red_score
    )
    /
    2.0

    -

    brown_score
)


hub18_unit = (

    adata_pb_centered.obs.iloc[
        eligible_indices
    ][
        [
            "patient",
            "Final_state_id",
            "N_cells"
        ]
    ]
    .copy()
)


hub18_unit[
    "Hub18_Blue"
] = blue_score


hub18_unit[
    "Hub18_Red"
] = red_score


hub18_unit[
    "Hub18_Brown"
] = brown_score


hub18_unit[
    "Hub18_BulkRisk_Concordance"
] = hub18_concordance


HUB18_UNIT_PATH = (

    RESULTS_DIR
    / "Step24A_Hub18_PatientState_Projection.csv"
)


hub18_unit.to_csv(
    HUB18_UNIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    hub18_unit
    .round(3)
)

In [ ]:
# ============================================================
# STEP24A - CELL 15
# Hub18 state-level summary
# ============================================================

hub18_state_summary = (

    hub18_unit

    .groupby(
        "Final_state_id"
    )

    .agg(

        N_patients=(
            "patient",
            "size"
        ),

        Blue_mean=(
            "Hub18_Blue",
            "mean"
        ),

        Red_mean=(
            "Hub18_Red",
            "mean"
        ),

        Brown_mean=(
            "Hub18_Brown",
            "mean"
        ),

        BulkRisk_concordance_mean=(
            "Hub18_BulkRisk_Concordance",
            "mean"
        ),

        BulkRisk_concordance_median=(
            "Hub18_BulkRisk_Concordance",
            "median"
        ),

        BulkRisk_concordance_sd=(
            "Hub18_BulkRisk_Concordance",
            "std"
        )

    )

    .reindex(
        STATE_ORDER
    )

    .reset_index()
)


HUB18_STATE_PATH = (

    RESULTS_DIR
    / "Step24A_Hub18_StateSummary.csv"
)


hub18_state_summary.to_csv(
    HUB18_STATE_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 105)
print("HUB18 BULK-RISK CONCORDANCE")
print("=" * 105)


display(
    hub18_state_summary
    .round(3)
)

In [ ]:
# ============================================================
# STEP24A - CELL 16
# Hub18 projection heatmap
# ============================================================

hub_plot = (

    hub18_state_summary

    .set_index(
        "Final_state_id"
    )[
        [
            "Blue_mean",
            "Red_mean",
            "Brown_mean",
            "BulkRisk_concordance_mean"
        ]
    ]

    .reindex(
        STATE_ORDER
    )
)


fig, ax = plt.subplots(
    figsize=(7.6, 6.2)
)


values = hub_plot.values


lim = np.nanmax(
    np.abs(
        values
    )
)


if not np.isfinite(lim) or lim == 0:
    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        hub_plot.shape[1]
    )
)


ax.set_xticklabels(

    [
        "Blue Module",
        "Red Module",
        "Brown Module",
        "Bulk-risk Concordance"
    ],

    rotation=35,
    ha="right"
)


ax.set_yticks(
    np.arange(
        len(STATE_ORDER)
    )
)


ax.set_yticklabels(
    STATE_ORDER
)


ax.set_xlabel(
    "Frozen Bulk Anchor"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "Hub18 Projection Across Final Malignant States"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Patient-adjusted Standardized Score"
)


fig.tight_layout()


save_publication_figure(

    fig,

    "Step24A_03_Hub18_BulkRiskConcordance"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24A - CELL 17
# Direct StableBio checkpoint-gene projection
# ============================================================

checkpoint_genes = [

    gene

    for gene in [
        "TNFRSF4",
        "CD276"
    ]

    if gene in gene_to_idx
]


checkpoint_projection = (

    build_state_gene_projection(

        checkpoint_genes,

        "StableBio7_Checkpoint"
    )
)


CHECKPOINT_PATH = (

    RESULTS_DIR
    / "Step24A_StableBio_CheckpointGene_Projection.csv"
)


checkpoint_projection.to_csv(
    CHECKPOINT_PATH,
    index=False,
    encoding="utf-8-sig"
)


print(
    "Directly projected checkpoint genes:",
    checkpoint_genes
)


display(
    checkpoint_projection
    .round(3)
)

In [ ]:
# ============================================================
# STEP24A - CELL 18
# Build state-level projection evidence table
# ============================================================

final_support[
    "Final_state_id"
] = (

    final_support[
        "Final_state_id"
    ]
    .astype(str)
)


projection_evidence = (

    hub18_state_summary

    .merge(

        final_support[
            [
                "Final_state_id",
                "N_eligible_patients",
                "Support_class"
            ]
        ],

        on=
            "Final_state_id",

        how=
            "left"
    )
)


original_detected = [

    gene
    for gene in ORIGINAL_FINAL_TOP10
    if gene in adata.var_names
]


stable_detected = [

    gene
    for gene in STABLE_GENES10
    if gene in adata.var_names
]


projection_evidence[
    "OriginalTop10_detected_n"
] = len(
    original_detected
)


projection_evidence[
    "StableGenes10_detected_n"
] = len(
    stable_detected
)


projection_evidence[
    "Hub18_detected_n"
] = len(
    hub_genes
)


projection_evidence[
    "Original_RiskScore_reconstructed"
] = False


projection_evidence[
    "Interpretation_status"
] = (
    "Projection_only_pending_convergence_review"
)


PROJECTION_EVIDENCE_PATH = (

    RESULTS_DIR
    / "Step24A_BulkProjection_StateEvidence.csv"
)


projection_evidence.to_csv(
    PROJECTION_EVIDENCE_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    projection_evidence
    .round(3)
)

In [ ]:
# ============================================================
# STEP24A - CELL 19
# Methodology / decision log
# ============================================================

step24a_summary = {

    "Dataset_scRNA":
        "GSE149614",

    "Bulk_reference":
        "TCGA-LIHC frozen bulk analyses",

    "Step":
        "Step24A_Frozen_Bulk_GeneAnchor_Projection",

    "Status":
        "PROJECTION_COMPLETE_PENDING_INTERPRETATION",

    "Final_malignant_states_modified":
        False,

    "Cross_cohort_design":
        (
            "TCGA-LIHC bulk and GSE149614 scRNA are independent "
            "cohorts. Projection is signature-based rather than "
            "direct patient-level RiskScore correlation."
        ),

    "Original_Final_Top10_defined":
        len(
            ORIGINAL_FINAL_TOP10
        ),

    "Original_Final_Top10_detected":
        len(
            original_detected
        ),

    "Original_Final_Top10_detected_genes":
        original_detected,

    "Original_RiskScore_reconstructed":
        False,

    "Reason_RiskScore_not_reconstructed":
        (
            "The frozen Original Final Top10 is incompletely "
            "represented in the scRNA dataset. An incomplete "
            "surrogate must not be labeled as Original RiskScore."
        ),

    "StableGenes10_defined":
        len(
            STABLE_GENES10
        ),

    "StableGenes10_detected":
        len(
            stable_detected
        ),

    "Hub18_detected":
        len(
            hub_genes
        ),

    "Hub18_concordance_definition":
        (
            "After within-patient centering and per-gene "
            "standardization: mean(Blue, Red) - Brown."
        ),

    "StableBio7_strategy":
        (
            "Only directly mappable checkpoint genes are projected "
            "here. Immune-cell abundance and pathway-level features "
            "are not converted into arbitrary malignant-cell scores."
        ),

    "Next_step":
        (
            "Review whether Original Final Top10 detected genes, "
            "StableGenes10 and Hub18 independently converge on "
            "specific frozen malignant states."
        )
}


SUMMARY_PATH = (

    LOGS_DIR
    / "Step24A_Final_Summary.json"
)


with open(
    SUMMARY_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        step24a_summary,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    SUMMARY_PATH
)

In [ ]:
# ============================================================
# STEP24A - CELL 20
# Output manifest + FINAL REVIEW
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR),
    ("Reference", REFERENCE_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_PATH = (

    RESULTS_DIR
    / "Step24A_Output_Manifest.csv"
)


manifest_df.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig"
)


print("=" * 125)
print("STEP24A FINAL BULK-TO-SINGLE-CELL PROJECTION REVIEW")
print("=" * 125)


print("\n[1] Frozen bulk-anchor coverage")
print("-" * 125)

display(
    coverage_table[
        [
            "Anchor",
            "N_defined",
            "N_detected",
            "Coverage_fraction",
            "Detected_genes",
            "Missing_genes"
        ]
    ]
)


print("\n[2] StableBio7 mapping audit")
print("-" * 125)

display(
    stablebio_mapping
)


print("\n[3] Original Final Top10 projection")
print("-" * 125)

display(
    original_top10_projection
    .round(3)
)


print("\n[4] StableGenes10 projection")
print("-" * 125)

display(
    stablegenes10_projection
    .round(3)
)


print("\n[5] Hub18 state projection")
print("-" * 125)

display(
    hub18_state_summary
    .round(3)
)


print("\n[6] StableBio checkpoint projection")
print("-" * 125)

display(
    checkpoint_projection
    .round(3)
)


print("\n[7] State-level projection evidence")
print("-" * 125)

display(
    projection_evidence
    .round(3)
)


print("\n[8] Files")
print("-" * 125)


required_files = [

    ANCHOR_JSON_PATH,
    COVERAGE_PATH,
    STABLEBIO_PATH,
    PB_LOG_PATH,
    CENTERED_PATH,

    ORIGINAL_PROJECTION_PATH,
    STABLE_PROJECTION_PATH,

    HUB18_UNIT_PATH,
    HUB18_STATE_PATH,

    CHECKPOINT_PATH,
    PROJECTION_EVIDENCE_PATH,

    SUMMARY_PATH,
    MANIFEST_PATH
]


for path in required_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP24A STATUS = FROZEN BULK GENE-ANCHOR PROJECTION COMPLETE"
)

print(
    "ORIGINAL RISK SCORE RECONSTRUCTED = NO"
)

print(
    "FINAL MALIGNANT STATES MODIFIED = NO"
)

print(
    "NEXT = BULK HIGH-RISK BIOLOGY CONVERGENCE ACROSS FINAL STATES"
)

print("=" * 125)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP24B - CELL 21
# Load frozen Step23 states + Step24A projection results
# ============================================================

from pathlib import Path

import json
import warnings

import numpy as np
import pandas as pd

import scanpy as sc

import matplotlib.pyplot as plt
import matplotlib as mpl

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)


# ============================================================
# Paths
# ============================================================

STEP23_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

STEP24_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step24_Bulk_to_scRNA_Projection')
)


DATA_DIR = STEP24_ROOT / "Data"
RESULTS_DIR = STEP24_ROOT / "Results"
FIGURES_DIR = STEP24_ROOT / "Figures"
LOGS_DIR = STEP24_ROOT / "Logs"


FINAL_STATE_INPUT = (
    STEP23_ROOT
    / "Data"
    / "GSE149614_STEP23_FINAL_MALIGNANT_STATES.h5ad"
)

HUB18_UNIT_INPUT = (
    RESULTS_DIR
    / "Step24A_Hub18_PatientState_Projection.csv"
)

STABLEGENES_INPUT = (
    RESULTS_DIR
    / "Step24A_StableGenes10_GeneProjection.csv"
)

STATE_SUPPORT_INPUT = (
    STEP23_ROOT
    / "Results"
    / "Step23_Final_State_PatientSupport.csv"
)

STATE_SUMMARY_INPUT = (
    STEP23_ROOT
    / "Results"
    / "Step23_Final_MalignantState_Summary.csv"
)


for path in [
    FINAL_STATE_INPUT,
    HUB18_UNIT_INPUT,
    STABLEGENES_INPUT,
    STATE_SUPPORT_INPUT,
    STATE_SUMMARY_INPUT
]:

    assert path.exists(), f"Missing:\n{path}"


adata = sc.read_h5ad(
    FINAL_STATE_INPUT
)

hub18_unit = pd.read_csv(
    HUB18_UNIT_INPUT
)

state_support = pd.read_csv(
    STATE_SUPPORT_INPUT
)

state_summary = pd.read_csv(
    STATE_SUMMARY_INPUT
)


STATE_KEY = (
    "final_malignant_state_id"
)

STATE_ORDER = [
    f"MS{i}"
    for i in range(1, 9)
]


print("=" * 115)
print("STEP24B BULK HIGH-RISK BIOLOGY CONVERGENCE")
print("=" * 115)

print(
    "Single-cell object:",
    adata.shape
)

print(
    "Eligible Hub18 patient-state units:",
    len(hub18_unit)
)

display(
    hub18_unit
    .head()
)

In [ ]:
# ============================================================
# STEP24B - CELL 22
# Define frozen bulk-aligned biological directions
# ============================================================

# Directly aligned high-risk biology
HIGH_RISK_DIRECT_PROGRAMS = [

    "Proliferation_G2M",
    "Stress_UPR",
    "Wnt_BetaCatenin_Target"

]


# Directly aligned opposite / metabolic biology
LOW_RISK_DIRECT_PROGRAMS = [

    "Fatty_Acid_Metabolism",
    "Bile_Acid_Metabolism"

]


# Contextual programs: shown but NOT used in the primary index
CONTEXTUAL_PROGRAMS = [

    "DNA_Replication_S",
    "Hepatic_Differentiation",
    "Hypoxia",
    "Glycolysis",
    "Progenitor_Epithelial",
    "Fetal_HCC_Program",
    "EMT_ECM",
    "TGFb_Response",
    "Inflammatory_NFkB"

]


bulk_biology_definition = {

    "High_risk_direct":
        HIGH_RISK_DIRECT_PROGRAMS,

    "Opposing_metabolic_direct":
        LOW_RISK_DIRECT_PROGRAMS,

    "Contextual_only":
        CONTEXTUAL_PROGRAMS,

    "Primary_convergence_index":
        (
            "Equal-weight mean of standardized Hub18 concordance, "
            "high-risk direct program block, and inverse metabolic block."
        ),

    "Important":
        (
            "This is a descriptive cross-cohort biological convergence "
            "index, NOT the Original RiskScore and NOT a prognostic model."
        )
}


BIOLOGY_JSON = (
    LOGS_DIR
    / "Step24B_BulkBiology_Definition.json"
)


with open(
    BIOLOGY_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        bulk_biology_definition,
        f,
        ensure_ascii=False,
        indent=4
    )


print(
    BIOLOGY_JSON
)

In [ ]:
# ============================================================
# STEP24B - CELL 23
# Audit biological program availability
# ============================================================

required_programs = (

    HIGH_RISK_DIRECT_PROGRAMS
    +
    LOW_RISK_DIRECT_PROGRAMS
    +
    CONTEXTUAL_PROGRAMS
)


program_audit_rows = []


for program in required_programs:

    column = (
        "Program__"
        +
        program
    )


    program_audit_rows.append({

        "Program":
            program,

        "Column":
            column,

        "Available":
            column in adata.obs.columns,

        "Role":
            (
                "High-risk direct"
                if program in HIGH_RISK_DIRECT_PROGRAMS

                else

                "Opposing metabolic direct"
                if program in LOW_RISK_DIRECT_PROGRAMS

                else

                "Contextual"
            )
    })


program_audit = pd.DataFrame(
    program_audit_rows
)


PROGRAM_AUDIT_PATH = (
    RESULTS_DIR
    / "Step24B_Program_Availability_Audit.csv"
)


program_audit.to_csv(
    PROGRAM_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    program_audit
)


assert program_audit[
    program_audit[
        "Role"
    ]
    !=
    "Contextual"
][
    "Available"
].all(), (
    "A direct bulk-aligned program is missing."
)

In [ ]:
# ============================================================
# STEP24B - CELL 24
# Patient × final-state biological program medians
# ============================================================

program_columns = [

    "Program__"
    +
    program

    for program in required_programs

    if (
        "Program__"
        +
        program
    )
    in adata.obs.columns
]


program_obs = (

    adata.obs[
        [
            "patient",
            STATE_KEY
        ]
        +
        program_columns
    ]
    .copy()
)


program_obs[
    "patient"
] = (
    program_obs[
        "patient"
    ]
    .astype(str)
)


program_obs[
    STATE_KEY
] = (
    program_obs[
        STATE_KEY
    ]
    .astype(str)
)


patient_state_program = (

    program_obs

    .groupby(
        [
            "patient",
            STATE_KEY
        ],
        observed=True
    )[
        program_columns
    ]

    .median()

    .reset_index()
)


cell_counts = (

    adata.obs

    .assign(

        patient_string=
            adata.obs[
                "patient"
            ]
            .astype(str),

        state_string=
            adata.obs[
                STATE_KEY
            ]
            .astype(str)
    )

    .groupby(
        [
            "patient_string",
            "state_string"
        ]
    )

    .size()

    .rename(
        "N_cells"
    )

    .reset_index()
)


cell_counts.columns = [

    "patient",
    STATE_KEY,
    "N_cells"

]


patient_state_program = (

    patient_state_program

    .merge(

        cell_counts,

        on=[
            "patient",
            STATE_KEY
        ],

        how="left",

        validate=
            "one_to_one"
    )
)


print(
    "Patient-state units:",
    len(
        patient_state_program
    )
)


display(
    patient_state_program
    .sort_values(
        [
            STATE_KEY,
            "patient"
        ]
    )
    .head(30)
)

In [ ]:
# ============================================================
# STEP24B - CELL 25
# Within-patient program deltas
# ============================================================

program_delta_rows = []


for _, row in (
    patient_state_program
    .iterrows()
):

    patient = str(
        row[
            "patient"
        ]
    )


    state = str(
        row[
            STATE_KEY
        ]
    )


    other_states = (

        patient_state_program[
            (
                patient_state_program[
                    "patient"
                ]
                .astype(str)
                ==
                patient
            )

            &

            (
                patient_state_program[
                    STATE_KEY
                ]
                .astype(str)
                !=
                state
            )
        ]
    )


    output = {

        "patient":
            patient,

        "Final_state_id":
            state,

        "N_cells":
            int(
                row[
                    "N_cells"
                ]
            )
    }


    for column in program_columns:

        clean_name = (
            column.replace(
                "Program__",
                ""
            )
        )


        if len(
            other_states
        ) == 0:

            value = np.nan

        else:

            value = (

                float(
                    row[
                        column
                    ]
                )

                -

                float(
                    other_states[
                        column
                    ]
                    .mean()
                )
            )


        output[
            clean_name
        ] = value


    program_delta_rows.append(
        output
    )


program_delta = pd.DataFrame(
    program_delta_rows
)


PROGRAM_DELTA_PATH = (
    RESULTS_DIR
    / "Step24B_PatientState_ProgramDelta.csv"
)


program_delta.to_csv(
    PROGRAM_DELTA_PATH,
    index=False,
    encoding="utf-8-sig"
)


display(
    program_delta
    .head(30)
    .round(4)
)

In [ ]:
# ============================================================
# STEP24B - CELL 26
# Eligible patient-state units
# ============================================================

MIN_CELLS_PER_PATIENT_STATE = 20


program_delta_eligible = (

    program_delta[
        program_delta[
            "N_cells"
        ]
        >=
        MIN_CELLS_PER_PATIENT_STATE
    ]
    .copy()
)


print(
    "Eligible biological-program units:",
    len(
        program_delta_eligible
    )
)


display(

    program_delta_eligible[
        [
            "patient",
            "Final_state_id",
            "N_cells"
        ]
    ]

    .sort_values(
        [
            "Final_state_id",
            "patient"
        ]
    )
)

In [ ]:
# ============================================================
# STEP24B - CELL 27
# Merge Hub18 and biological-program evidence
# ============================================================

hub18_merge = (

    hub18_unit[
        [
            "patient",
            "Final_state_id",
            "N_cells",
            "Hub18_Blue",
            "Hub18_Red",
            "Hub18_Brown",
            "Hub18_BulkRisk_Concordance"
        ]
    ]
    .copy()
)


hub18_merge[
    "patient"
] = (
    hub18_merge[
        "patient"
    ]
    .astype(str)
)


hub18_merge[
    "Final_state_id"
] = (
    hub18_merge[
        "Final_state_id"
    ]
    .astype(str)
)


merged_units = (

    program_delta_eligible

    .merge(

        hub18_merge,

        on=[
            "patient",
            "Final_state_id"
        ],

        how="inner",

        suffixes=(
            "_program",
            "_hub"
        ),

        validate=
            "one_to_one"
    )
)


assert (
    len(
        merged_units
    )
    ==
    len(
        hub18_merge
    )
), (
    "Hub18 and program eligible units do not match."
)


print(
    "Merged eligible units:",
    len(
        merged_units
    )
)


display(
    merged_units
    .head()
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 27
# Merge Hub18 and biological-program evidence
# FIXED: reconcile duplicated N_cells columns
# ============================================================

hub18_merge = (

    hub18_unit[
        [
            "patient",
            "Final_state_id",
            "N_cells",
            "Hub18_Blue",
            "Hub18_Red",
            "Hub18_Brown",
            "Hub18_BulkRisk_Concordance"
        ]
    ]
    .copy()
)


hub18_merge[
    "patient"
] = (
    hub18_merge[
        "patient"
    ]
    .astype(str)
)


hub18_merge[
    "Final_state_id"
] = (
    hub18_merge[
        "Final_state_id"
    ]
    .astype(str)
)


program_delta_eligible[
    "patient"
] = (
    program_delta_eligible[
        "patient"
    ]
    .astype(str)
)


program_delta_eligible[
    "Final_state_id"
] = (
    program_delta_eligible[
        "Final_state_id"
    ]
    .astype(str)
)


# ============================================================
# Merge
# ============================================================

merged_units = (

    program_delta_eligible

    .merge(

        hub18_merge,

        on=[
            "patient",
            "Final_state_id"
        ],

        how="inner",

        suffixes=(
            "_program",
            "_hub"
        ),

        validate=
            "one_to_one"
    )
)


# ============================================================
# Audit: all eligible Hub18 units must match
# ============================================================

assert (
    len(merged_units)
    ==
    len(hub18_merge)
), (
    "Hub18 and program eligible units do not match."
)


# ============================================================
# Reconcile duplicated N_cells
# Both tables should describe exactly the same patient-state unit
# ============================================================

assert (
    merged_units[
        "N_cells_program"
    ]
    .astype(int)
    .values

    ==

    merged_units[
        "N_cells_hub"
    ]
    .astype(int)
    .values
).all(), (
    "N_cells mismatch between program and Hub18 tables."
)


merged_units[
    "N_cells"
] = (

    merged_units[
        "N_cells_program"
    ]
    .astype(int)
)


# Keep audit columns if desired, but move canonical N_cells forward
preferred_columns = [

    "patient",
    "Final_state_id",
    "N_cells",
    "N_cells_program",
    "N_cells_hub"

]


other_columns = [

    col
    for col in merged_units.columns
    if col not in preferred_columns

]


merged_units = merged_units[
    preferred_columns
    +
    other_columns
]


print("=" * 110)
print("HUB18 × BIOLOGICAL PROGRAM MERGE AUDIT")
print("=" * 110)

print(
    "Program eligible units:",
    len(program_delta_eligible)
)

print(
    "Hub18 eligible units:",
    len(hub18_merge)
)

print(
    "Merged eligible units:",
    len(merged_units)
)

print(
    "N_cells agreement = PASS"
)


display(

    merged_units[
        [
            "patient",
            "Final_state_id",
            "N_cells",
            "Hub18_Blue",
            "Hub18_Red",
            "Hub18_Brown",
            "Hub18_BulkRisk_Concordance"
        ]
    ]
    .head(20)
    .round(3)

)

In [ ]:
# ============================================================
# STEP24B - CELL 28
# Standardize independent evidence blocks
# ============================================================

def safe_zscore(
    series
):

    values = (
        series
        .astype(float)
    )


    mean = (
        values.mean()
    )


    sd = (
        values.std(
            ddof=1
        )
    )


    if (
        not np.isfinite(sd)
        or
        sd == 0
    ):

        return pd.Series(

            np.zeros(
                len(values)
            ),

            index=
                values.index
        )


    return (
        values
        -
        mean
    ) / sd


# ============================================================
# Hub18
# ============================================================

merged_units[
    "Z_Hub18"
] = safe_zscore(

    merged_units[
        "Hub18_BulkRisk_Concordance"
    ]
)


# ============================================================
# Individual biological programs
# ============================================================

for program in (

    HIGH_RISK_DIRECT_PROGRAMS
    +
    LOW_RISK_DIRECT_PROGRAMS
    +
    CONTEXTUAL_PROGRAMS

):

    if program not in merged_units.columns:
        continue


    merged_units[
        f"Z_{program}"
    ] = safe_zscore(

        merged_units[
            program
        ]
    )


# ============================================================
# High-risk direct program block
# ============================================================

highrisk_z_cols = [

    f"Z_{program}"

    for program
    in HIGH_RISK_DIRECT_PROGRAMS
]


merged_units[
    "HighRisk_DirectProgram_Block"
] = (

    merged_units[
        highrisk_z_cols
    ]
    .mean(
        axis=1
    )
)


# ============================================================
# Opposing metabolic block
# ============================================================

lowrisk_z_cols = [

    f"Z_{program}"

    for program
    in LOW_RISK_DIRECT_PROGRAMS
]


merged_units[
    "Opposing_Metabolic_Block"
] = (

    merged_units[
        lowrisk_z_cols
    ]
    .mean(
        axis=1
    )
)


# ============================================================
# PRIMARY descriptive convergence index
#
# Positive:
#   bulk high-risk biology concordant
#
# Negative:
#   bulk opposing/metabolic biology concordant
#
# NOT a RiskScore.
# ============================================================

merged_units[
    "BulkRisk_Biology_Convergence"
] = (

    (
        merged_units[
            "Z_Hub18"
        ]

        +

        merged_units[
            "HighRisk_DirectProgram_Block"
        ]

        -

        merged_units[
            "Opposing_Metabolic_Block"
        ]
    )

    /
    3.0
)


UNIT_CONVERGENCE_PATH = (

    RESULTS_DIR
    / "Step24B_PatientState_BulkRiskBiology_Convergence.csv"
)


merged_units.to_csv(

    UNIT_CONVERGENCE_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    merged_units[
        [
            "patient",
            "Final_state_id",
            "N_cells",
            "Z_Hub18",
            "HighRisk_DirectProgram_Block",
            "Opposing_Metabolic_Block",
            "BulkRisk_Biology_Convergence"
        ]
    ]
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 29
# State-level convergence summary
# ============================================================

state_convergence = (

    merged_units

    .groupby(
        "Final_state_id"
    )

    .agg(

        N_patients=(
            "patient",
            "size"
        ),

        Hub18_mean=(
            "Z_Hub18",
            "mean"
        ),

        HighRiskProgram_mean=(
            "HighRisk_DirectProgram_Block",
            "mean"
        ),

        OpposingMetabolic_mean=(
            "Opposing_Metabolic_Block",
            "mean"
        ),

        Convergence_mean=(
            "BulkRisk_Biology_Convergence",
            "mean"
        ),

        Convergence_median=(
            "BulkRisk_Biology_Convergence",
            "median"
        ),

        Convergence_sd=(
            "BulkRisk_Biology_Convergence",
            "std"
        ),

        Positive_fraction=(
            "BulkRisk_Biology_Convergence",

            lambda x:
                float(
                    np.mean(
                        x > 0
                    )
                )
        )

    )

    .reindex(
        STATE_ORDER
    )

    .reset_index()
)


STATE_CONVERGENCE_PATH = (

    RESULTS_DIR
    / "Step24B_State_BulkRiskBiology_Convergence.csv"
)


state_convergence.to_csv(

    STATE_CONVERGENCE_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 110)
print("STATE-LEVEL BULK-RISK BIOLOGY CONVERGENCE")
print("=" * 110)


display(
    state_convergence
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 30
# Contextual biological evidence
# ============================================================

context_rows = []


for state in STATE_ORDER:

    temp = (

        merged_units[
            merged_units[
                "Final_state_id"
            ]
            ==
            state
        ]
    )


    row = {

        "Final_state_id":
            state,

        "N_patients":
            len(temp)
    }


    for program in CONTEXTUAL_PROGRAMS:

        z_col = (
            f"Z_{program}"
        )


        if z_col in temp.columns:

            row[
                program
            ] = float(
                temp[
                    z_col
                ]
                .mean()
            )


    context_rows.append(
        row
    )


context_summary = pd.DataFrame(
    context_rows
)


CONTEXT_PATH = (

    RESULTS_DIR
    / "Step24B_ContextualProgram_StateSummary.csv"
)


context_summary.to_csv(

    CONTEXT_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    context_summary
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 31
# StableGenes10 ↔ Hub18 directional bridge genes
# ============================================================

stablegenes_projection = pd.read_csv(
    STABLEGENES_INPUT
)


BRIDGE_GENE_DIRECTION = {

    "PRR11":
        "High-risk direction (Hub18 Blue)",

    "BOP1":
        "High-risk direction (Hub18 Red)",

    "GLYATL1":
        "Opposing metabolic direction (Hub18 Brown)"

}


bridge_gene_table = (

    stablegenes_projection[
        stablegenes_projection[
            "Gene"
        ]
        .isin(
            BRIDGE_GENE_DIRECTION.keys()
        )
    ]
    .copy()
)


bridge_gene_table[
    "Frozen_bulk_direction"
] = (

    bridge_gene_table[
        "Gene"
    ]
    .map(
        BRIDGE_GENE_DIRECTION
    )
)


BRIDGE_PATH = (

    RESULTS_DIR
    / "Step24B_StableGenes_Hub18_BridgeGenes.csv"
)


bridge_gene_table.to_csv(

    BRIDGE_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    bridge_gene_table
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 32
# Cross-layer convergence heatmap
# ============================================================

heat_rows = []


for state in STATE_ORDER:

    temp = (

        merged_units[
            merged_units[
                "Final_state_id"
            ]
            ==
            state
        ]
    )


    heat_rows.append({

        "Final_state_id":
            state,

        "Hub18":
            temp[
                "Z_Hub18"
            ].mean(),

        "G2M":
            temp[
                "Z_Proliferation_G2M"
            ].mean(),

        "UPR":
            temp[
                "Z_Stress_UPR"
            ].mean(),

        "Wnt":
            temp[
                "Z_Wnt_BetaCatenin_Target"
            ].mean(),

        "Inverse Fatty Acid":
            -temp[
                "Z_Fatty_Acid_Metabolism"
            ].mean(),

        "Inverse Bile Acid":
            -temp[
                "Z_Bile_Acid_Metabolism"
            ].mean(),

        "Convergence":
            temp[
                "BulkRisk_Biology_Convergence"
            ].mean()
    })


convergence_heatmap = (

    pd.DataFrame(
        heat_rows
    )

    .set_index(
        "Final_state_id"
    )
)


HEATMAP_TABLE_PATH = (

    RESULTS_DIR
    / "Step24B_CrossLayer_Convergence_HeatmapData.csv"
)


convergence_heatmap.to_csv(
    HEATMAP_TABLE_PATH,
    encoding="utf-8-sig"
)


fig, ax = plt.subplots(
    figsize=(9.2, 6.3)
)


values = (
    convergence_heatmap.values
)


lim = np.nanmax(
    np.abs(
        values
    )
)


if (
    not np.isfinite(lim)
    or
    lim == 0
):

    lim = 1


im = ax.imshow(

    values,

    cmap="RdBu_r",

    vmin=-lim,
    vmax=lim,

    aspect="auto"
)


ax.set_xticks(
    np.arange(
        convergence_heatmap.shape[1]
    )
)


ax.set_xticklabels(

    convergence_heatmap.columns,

    rotation=40,

    ha="right"
)


ax.set_yticks(
    np.arange(
        len(STATE_ORDER)
    )
)


ax.set_yticklabels(
    STATE_ORDER
)


ax.set_xlabel(
    "Bulk-aligned Evidence Axis"
)

ax.set_ylabel(
    "Final Malignant State"
)

ax.set_title(
    "Cross-layer Convergence of Bulk Risk Biology and Malignant States"
)


cbar = fig.colorbar(
    im,
    ax=ax
)


cbar.set_label(
    "Standardized Directional Evidence"
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR
    / "Step24B_01_CrossLayer_BulkRiskConvergence.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR
    / "Step24B_01_CrossLayer_BulkRiskConvergence.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24B - CELL 33
# Patient-state convergence distribution
# ============================================================

fig, ax = plt.subplots(
    figsize=(8.8, 5.8)
)


x_positions = {

    state: i

    for i, state
    in enumerate(
        STATE_ORDER
    )
}


patients = sorted(
    merged_units[
        "patient"
    ]
    .unique()
)


patient_markers = [
    "o",
    "s",
    "^",
    "D",
    "P"
]


for p_idx, patient in enumerate(
    patients
):

    temp = (

        merged_units[
            merged_units[
                "patient"
            ]
            ==
            patient
        ]
    )


    x = [

        x_positions[
            state
        ]

        for state in temp[
            "Final_state_id"
        ]
    ]


    y = (

        temp[
            "BulkRisk_Biology_Convergence"
        ]
        .values
    )


    ax.scatter(

        x,
        y,

        s=45,

        marker=
            patient_markers[
                p_idx
                %
                len(
                    patient_markers
                )
            ],

        alpha=0.85,

        label=
            patient
    )


# State means
for i, state in enumerate(
    STATE_ORDER
):

    temp = (

        merged_units[
            merged_units[
                "Final_state_id"
            ]
            ==
            state
        ][
            "BulkRisk_Biology_Convergence"
        ]
    )


    if len(temp) == 0:
        continue


    mean_value = (
        temp.mean()
    )


    ax.plot(

        [
            i - 0.22,
            i + 0.22
        ],

        [
            mean_value,
            mean_value
        ],

        linewidth=2.3,

        solid_capstyle=
            "round"
    )


ax.axhline(

    0,

    linewidth=0.9,

    linestyle="--"
)


ax.set_xticks(
    np.arange(
        len(
            STATE_ORDER
        )
    )
)


ax.set_xticklabels(
    STATE_ORDER
)


ax.set_xlabel(
    "Final Malignant State"
)

ax.set_ylabel(
    "Bulk-risk Biology Convergence Index"
)

ax.set_title(
    "Patient-level Consistency of Bulk-risk Biology Projection"
)


ax.legend(

    title=
        "Patient",

    frameon=
        False,

    bbox_to_anchor=
        (
            1.02,
            1
        ),

    loc=
        "upper left"
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR
    / "Step24B_02_PatientState_Convergence.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR
    / "Step24B_02_PatientState_Convergence.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24B - CELL 34
# Master convergence evidence table
# ============================================================

state_name_table = (

    adata.obs[
        [
            "final_malignant_state_id",
            "final_malignant_state",
            "final_state_family",
            "final_state_scope"
        ]
    ]

    .drop_duplicates()

    .copy()
)


state_name_table.columns = [

    "Final_state_id",
    "Final_state_name",
    "State_family",
    "State_scope"
]


state_support[
    "Final_state_id"
] = (
    state_support[
        "Final_state_id"
    ]
    .astype(str)
)


master_evidence = (

    state_name_table

    .merge(

        state_convergence,

        on=
            "Final_state_id",

        how=
            "left"
    )

    .merge(

        state_support[
            [
                "Final_state_id",
                "N_eligible_patients",
                "Support_class"
            ]
        ],

        on=
            "Final_state_id",

        how=
            "left"
    )

    .merge(

        context_summary,

        on=
            "Final_state_id",

        how=
            "left",

        suffixes=(
            "",
            "_context"
        )
    )
)


master_evidence[
    "Convergence_direction"
] = np.where(

    master_evidence[
        "Convergence_mean"
    ]
    >
    0,

    "Positive_bulk-risk_concordance",

    "Opposing_bulk-risk_concordance"
)


master_evidence[
    "Interpretation_caution"
] = np.where(

    master_evidence[
        "N_eligible_patients"
    ]
    <=
    1,

    "Patient-specific; do not generalize without external validation",

    "Cross-patient evidence available"
)


MASTER_EVIDENCE_PATH = (

    RESULTS_DIR
    / "Step24B_Master_BulkRisk_StateEvidence.csv"
)


master_evidence.to_csv(

    MASTER_EVIDENCE_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(
    master_evidence
    .round(3)
)

In [ ]:
# ============================================================
# STEP24B - CELL 35
# Methodological decision log
# ============================================================

step24b_summary = {

    "Dataset_scRNA":
        "GSE149614",

    "Bulk_reference":
        "TCGA-LIHC frozen bulk risk biology",

    "Step":
        "Step24B_BulkRisk_Biology_Convergence",

    "Status":
        "COMPLETE_PENDING_INTERPRETATION",

    "Final_states_modified":
        False,

    "Original_RiskScore_reconstructed":
        False,

    "Primary_evidence_blocks": {

        "Hub18":
            "Frozen directional WGCNA/hub-gene bulk anchor",

        "HighRisk_direct_programs":
            HIGH_RISK_DIRECT_PROGRAMS,

        "Opposing_metabolic_programs":
            LOW_RISK_DIRECT_PROGRAMS
    },

    "Contextual_programs_not_in_primary_index":
        CONTEXTUAL_PROGRAMS,

    "Convergence_index_definition":
        (
            "Equal-weight mean of standardized Hub18 bulk-risk "
            "concordance, high-risk direct malignant-cell program block, "
            "and inverse opposing metabolic program block."
        ),

    "Important_limitation":
        (
            "The convergence index is descriptive and cross-cohort. "
            "It is not a patient prognostic model, not a Cox RiskScore, "
            "and must not be reported as the Original RiskScore."
        ),

    "Patient_state_minimum_cells":
        int(
            MIN_CELLS_PER_PATIENT_STATE
        ),

    "Next_step":
        (
            "Review convergence results and determine which frozen "
            "malignant states are supported as bulk-risk-concordant "
            "versus bulk-risk-opposing states before Step25 "
            "cross-patient robustness analysis."
        )
}


SUMMARY_B_PATH = (

    LOGS_DIR
    / "Step24B_Final_Summary.json"
)


with open(

    SUMMARY_B_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step24b_summary,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    SUMMARY_B_PATH
)

In [ ]:
# ============================================================
# STEP24B - CELL 36
# Output manifest
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


MANIFEST_B_PATH = (

    RESULTS_DIR
    / "Step24B_Output_Manifest.csv"
)


manifest_df.to_csv(

    MANIFEST_B_PATH,

    index=False,
    encoding="utf-8-sig"
)


print(
    "Saved:",
    MANIFEST_B_PATH
)

In [ ]:
# ============================================================
# STEP24B - CELL 37
# FINAL REVIEW OUTPUT
# ============================================================

print("=" * 125)
print("STEP24B FINAL BULK-RISK BIOLOGY CONVERGENCE REVIEW")
print("=" * 125)


print("\n[1] Program availability")
print("-" * 125)

display(
    program_audit
)


print("\n[2] Eligible patient-state convergence units")
print("-" * 125)

display(

    merged_units[
        [
            "patient",
            "Final_state_id",
            "N_cells",
            "Z_Hub18",
            "HighRisk_DirectProgram_Block",
            "Opposing_Metabolic_Block",
            "BulkRisk_Biology_Convergence"
        ]
    ]

    .sort_values(
        [
            "Final_state_id",
            "patient"
        ]
    )

    .round(3)
)


print("\n[3] State-level convergence")
print("-" * 125)

display(
    state_convergence
    .round(3)
)


print("\n[4] Contextual programs")
print("-" * 125)

display(
    context_summary
    .round(3)
)


print("\n[5] StableGenes10-Hub18 bridge genes")
print("-" * 125)

display(
    bridge_gene_table
    .round(3)
)


print("\n[6] Master state evidence")
print("-" * 125)

display(
    master_evidence
    .round(3)
)


print("\n[7] Files")
print("-" * 125)


required_files = [

    BIOLOGY_JSON,
    PROGRAM_AUDIT_PATH,
    PROGRAM_DELTA_PATH,
    UNIT_CONVERGENCE_PATH,
    STATE_CONVERGENCE_PATH,
    CONTEXT_PATH,
    BRIDGE_PATH,
    HEATMAP_TABLE_PATH,
    MASTER_EVIDENCE_PATH,
    SUMMARY_B_PATH,
    MANIFEST_B_PATH

]


for path in required_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP24B STATUS = CROSS-LAYER CONVERGENCE COMPLETE"
)

print(
    "ORIGINAL RISK SCORE RECONSTRUCTED = NO"
)

print(
    "FINAL MALIGNANT STATES MODIFIED = NO"
)

print(
    "NEXT = REVIEW CONVERGENCE AND LOCK BULK-RISK-ASSOCIATED MALIGNANT STATES"
)

print("=" * 125)

In [ ]:
from hcc_release_paths import project_path as _hcc_path
# ============================================================
# STEP24 FINAL LOCK - CELL 38
# Define final bulk-risk concordance roles
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import scanpy as sc


STEP23_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step23_Malignant_State_Analysis')
)

STEP24_ROOT = Path(
    _hcc_path('LIHC_scRNA_Step24_Bulk_to_scRNA_Projection')
)

DATA_DIR = STEP24_ROOT / "Data"
RESULTS_DIR = STEP24_ROOT / "Results"
FIGURES_DIR = STEP24_ROOT / "Figures"
LOGS_DIR = STEP24_ROOT / "Logs"


MASTER_INPUT = (
    RESULTS_DIR /
    "Step24B_Master_BulkRisk_StateEvidence.csv"
)

FINAL_STATE_INPUT = (
    STEP23_ROOT /
    "Data" /
    "GSE149614_STEP23_FINAL_MALIGNANT_STATES.h5ad"
)


assert MASTER_INPUT.exists()
assert FINAL_STATE_INPUT.exists()


master_evidence = pd.read_csv(
    MASTER_INPUT
)

adata = sc.read_h5ad(
    FINAL_STATE_INPUT
)


RISK_ROLE_MAPPING = {

    "MS1": {
        "role":
            "Secondary_bulk-risk-concordant",

        "priority":
            "Secondary",

        "interpretation":
            (
                "Cross-patient positive concordance with prominent "
                "hypoxia, TGFb, EMT/ECM and inflammatory programs; "
                "patient-state evidence is less uniformly robust than MS3."
            )
    },

    "MS2": {
        "role":
            "Mixed_bulk-risk-opposing",

        "priority":
            "Contextual",

        "interpretation":
            (
                "Mean bulk-risk concordance is negative across four "
                "patients, but one patient shows positive concordance; "
                "therefore this state is not used as the primary "
                "risk-opposing reference."
            )
    },

    "MS3": {
        "role":
            "Primary_bulk-risk-concordant",

        "priority":
            "Primary_target",

        "interpretation":
            (
                "Strongest cross-patient convergence of Hub18, "
                "high-risk malignant-cell programs and inverse "
                "metabolic biology across three supported patients."
            )
    },

    "MS4": {
        "role":
            "Patient-specific_bulk-risk-opposing",

        "priority":
            "Patient_specific",

        "interpretation":
            (
                "HCC08-specific state with negative convergence; "
                "not generalized beyond the discovery patient."
            )
    },

    "MS5": {
        "role":
            "Supportive_bulk-risk-concordant",

        "priority":
            "Supportive",

        "interpretation":
            (
                "Two-patient positive concordance with progenitor "
                "and inflammatory features, but substantially weaker "
                "than MS3."
            )
    },

    "MS6": {
        "role":
            "Primary_bulk-risk-opposing",

        "priority":
            "Primary_reference",

        "interpretation":
            (
                "Strong and consistent negative bulk-risk concordance "
                "with marked hepatic differentiation and opposing "
                "metabolic programs across two supported patients."
            )
    },

    "MS7": {
        "role":
            "Patient-specific_bulk-risk-opposing",

        "priority":
            "Patient_specific",

        "interpretation":
            (
                "HCC04-specific negative convergence; retained as "
                "a patient-specific metabolic branch."
            )
    },

    "MS8": {
        "role":
            "Patient-specific_bulk-risk-opposing",

        "priority":
            "Patient_specific",

        "interpretation":
            (
                "HCC04-specific negative convergence with hepatic, "
                "fatty-acid and bile-acid metabolic features."
            )
    }
}


risk_role_table = pd.DataFrame(

    [
        {
            "Final_state_id":
                state,

            "Step24_role":
                info[
                    "role"
                ],

            "Step24_priority":
                info[
                    "priority"
                ],

            "Step24_interpretation":
                info[
                    "interpretation"
                ]
        }

        for state, info
        in RISK_ROLE_MAPPING.items()
    ]
)


risk_role_table = (

    risk_role_table

    .merge(

        master_evidence,

        on=
            "Final_state_id",

        how=
            "left",

        validate=
            "one_to_one"
    )
)


RISK_ROLE_PATH = (

    RESULTS_DIR /
    "Step24_FINAL_BulkRisk_StateRoles.csv"
)


risk_role_table.to_csv(

    RISK_ROLE_PATH,

    index=False,
    encoding="utf-8-sig"
)


display(

    risk_role_table[
        [
            "Final_state_id",
            "Final_state_name",
            "Step24_role",
            "N_eligible_patients",
            "Convergence_mean",
            "Convergence_median",
            "Positive_fraction",
            "Step24_interpretation"
        ]
    ]
    .round(3)

)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 39
# Primary risk-concordant vs risk-opposing axis
# ============================================================

PRIMARY_TARGET = "MS3"
PRIMARY_REFERENCE = "MS6"


axis_columns = [

    "Final_state_id",
    "Final_state_name",
    "N_eligible_patients",

    "Hub18_mean",
    "HighRiskProgram_mean",
    "OpposingMetabolic_mean",

    "Convergence_mean",
    "Convergence_median",
    "Positive_fraction",

    "DNA_Replication_S",
    "Hepatic_Differentiation",
    "Hypoxia",
    "Glycolysis",
    "Progenitor_Epithelial",
    "Fetal_HCC_Program",
    "EMT_ECM",
    "TGFb_Response",
    "Inflammatory_NFkB"
]


axis_table = (

    master_evidence[
        master_evidence[
            "Final_state_id"
        ]
        .isin(
            [
                PRIMARY_TARGET,
                PRIMARY_REFERENCE
            ]
        )
    ][
        axis_columns
    ]

    .set_index(
        "Final_state_id"
    )

    .reindex(
        [
            PRIMARY_TARGET,
            PRIMARY_REFERENCE
        ]
    )

    .reset_index()
)


AXIS_PATH = (

    RESULTS_DIR /
    "Step24_FINAL_MS3_vs_MS6_PrimaryAxis.csv"
)


axis_table.to_csv(

    AXIS_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 110)
print("PRIMARY CROSS-COHORT MALIGNANT-STATE AXIS")
print("=" * 110)

display(
    axis_table
    .round(3)
)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 40
# Annotate FROZEN Step23 states with Step24 risk-concordance roles
# Do NOT overwrite Step23
# ============================================================

role_lookup = {

    state:
        info[
            "role"
        ]

    for state, info
    in RISK_ROLE_MAPPING.items()
}


priority_lookup = {

    state:
        info[
            "priority"
        ]

    for state, info
    in RISK_ROLE_MAPPING.items()
}


STATE_KEY = (
    "final_malignant_state_id"
)


adata.obs[
    "Step24_bulkRisk_role"
] = (

    adata.obs[
        STATE_KEY
    ]
    .astype(str)
    .map(
        role_lookup
    )
)


adata.obs[
    "Step24_bulkRisk_priority"
] = (

    adata.obs[
        STATE_KEY
    ]
    .astype(str)
    .map(
        priority_lookup
    )
)


assert (
    adata.obs[
        "Step24_bulkRisk_role"
    ]
    .isna()
    .sum()
    ==
    0
)


adata.uns[
    "Step24_bulkRisk_projection"
] = {

    "status":
        "FROZEN",

    "primary_bulk_risk_concordant_state":
        "MS3",

    "secondary_bulk_risk_concordant_state":
        "MS1",

    "supportive_bulk_risk_concordant_state":
        "MS5",

    "primary_bulk_risk_opposing_state":
        "MS6",

    "important":
        (
            "These annotations represent cross-cohort biological "
            "concordance with frozen TCGA bulk risk biology. "
            "They are not patient-level prognostic labels and "
            "do not reconstruct the Original RiskScore."
        )
}


STEP24_ANNOTATED_H5AD = (

    DATA_DIR /
    "GSE149614_STEP24_BULKRISK_ANNOTATED_MALIGNANT_STATES.h5ad"
)


adata.write_h5ad(

    STEP24_ANNOTATED_H5AD,

    compression="gzip"
)


print(
    "Saved:",
    STEP24_ANNOTATED_H5AD
)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 41
# Frozen UMAP with Step24 bulk-risk concordance roles
# ============================================================

ROLE_ORDER = [

    "Primary_bulk-risk-concordant",
    "Secondary_bulk-risk-concordant",
    "Supportive_bulk-risk-concordant",
    "Mixed_bulk-risk-opposing",
    "Primary_bulk-risk-opposing",
    "Patient-specific_bulk-risk-opposing"
]


adata.obs[
    "Step24_bulkRisk_role"
] = pd.Categorical(

    adata.obs[
        "Step24_bulkRisk_role"
    ],

    categories=
        ROLE_ORDER,

    ordered=True
)


ROLE_COLORS = {

    "Primary_bulk-risk-concordant":
        "#E64B35",

    "Secondary_bulk-risk-concordant":
        "#F39B7F",

    "Supportive_bulk-risk-concordant":
        "#DC0000",

    "Mixed_bulk-risk-opposing":
        "#B09C85",

    "Primary_bulk-risk-opposing":
        "#4DBBD5",

    "Patient-specific_bulk-risk-opposing":
        "#8491B4"
}


ax = sc.pl.embedding(

    adata,

    basis=
        "umap_lineage_clean",

    color=
        "Step24_bulkRisk_role",

    palette=
        ROLE_COLORS,

    size=
        8,

    alpha=
        0.82,

    frameon=
        False,

    legend_loc=
        "right margin",

    title=
        "Cross-cohort Bulk-risk Concordance Across Malignant States",

    show=
        False
)


fig = ax.figure

fig.set_size_inches(
    9.5,
    6.7
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step24_FINAL_01_BulkRiskConcordance_UMAP.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step24_FINAL_01_BulkRiskConcordance_UMAP.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 42
# MS3 vs MS6 cross-layer evidence
# ============================================================

plot_axes = [

    "Hub18_mean",
    "HighRiskProgram_mean",
    "OpposingMetabolic_mean",
    "Convergence_mean",
    "DNA_Replication_S",
    "Hepatic_Differentiation"
]


plot_labels = [

    "Hub18",
    "High-risk Programs",
    "Opposing Metabolism",
    "Overall Convergence",
    "DNA Replication",
    "Hepatic Differentiation"
]


target_values = (

    axis_table[
        axis_table[
            "Final_state_id"
        ]
        ==
        "MS3"
    ][
        plot_axes
    ]
    .iloc[0]
    .astype(float)
    .values
)


reference_values = (

    axis_table[
        axis_table[
            "Final_state_id"
        ]
        ==
        "MS6"
    ][
        plot_axes
    ]
    .iloc[0]
    .astype(float)
    .values
)


x = np.arange(
    len(
        plot_axes
    )
)


width = 0.36


fig, ax = plt.subplots(
    figsize=(10.0, 5.8)
)


ax.bar(

    x - width / 2,

    target_values,

    width,

    label=
        "MS3 Cycling-G2M"
)


ax.bar(

    x + width / 2,

    reference_values,

    width,

    label=
        "MS6 Hepatic-Differentiated"
)


ax.axhline(

    0,

    linewidth=0.8
)


ax.set_xticks(
    x
)


ax.set_xticklabels(

    plot_labels,

    rotation=35,

    ha="right"
)


ax.set_ylabel(
    "Standardized / Patient-adjusted Evidence"
)


ax.set_xlabel(
    "Cross-layer Biological Evidence"
)


ax.set_title(
    "Primary Bulk-risk-concordant and Opposing Malignant-state Axis"
)


ax.legend(
    frameon=False
)


fig.tight_layout()


fig.savefig(

    FIGURES_DIR /
    "Step24_FINAL_02_MS3_vs_MS6_PrimaryAxis.png",

    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)


fig.savefig(

    FIGURES_DIR /
    "Step24_FINAL_02_MS3_vs_MS6_PrimaryAxis.pdf",

    bbox_inches="tight",
    facecolor="white"
)


plt.show()
plt.close(fig)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 43
# Final Step24 decision record
# ============================================================

step24_final_decision = {

    "Step":
        "Step24_Bulk_to_scRNA_Projection",

    "Status":
        "FROZEN",

    "Cross_cohort_design":
        True,

    "Bulk_cohort":
        "TCGA-LIHC",

    "Single_cell_cohort":
        "GSE149614",

    "Original_RiskScore_reconstructed":
        False,

    "Malignant_states_redefined":
        False,

    "Primary_bulk_risk_concordant_state":
        {

            "State":
                "MS3",

            "Name":
                "Cycling-G2M",

            "Rationale":
                (
                    "Strongest cross-patient convergence of Hub18, "
                    "high-risk direct biological programs and inverse "
                    "metabolic programs across HCC08, HCC09 and HCC10."
                )
        },

    "Secondary_bulk_risk_concordant_state":
        {

            "State":
                "MS1",

            "Name":
                "Hypoxia-TGFb-Inflammatory",

            "Rationale":
                (
                    "Positive cross-patient convergence with strong "
                    "hypoxia, TGFb, EMT/ECM and inflammatory biology, "
                    "but less robust than MS3."
                )
        },

    "Supportive_bulk_risk_concordant_state":
        {

            "State":
                "MS5",

            "Name":
                "Stress-Inflammatory-Progenitor",

            "Rationale":
                (
                    "Positive but weaker two-patient convergence with "
                    "progenitor and inflammatory features."
                )
        },

    "Primary_bulk_risk_opposing_state":
        {

            "State":
                "MS6",

            "Name":
                "Hepatic-Differentiated",

            "Rationale":
                (
                    "Strong and consistent negative convergence with "
                    "marked hepatic differentiation and opposing "
                    "metabolic biology across HCC09 and HCC10."
                )
        },

    "Mixed_cross_patient_opposing_state":
        "MS2",

    "Patient_specific_states":
        [
            "MS4",
            "MS7",
            "MS8"
        ],

    "Primary_downstream_axis":
        "MS3 versus MS6",

    "Interpretation":
        (
            "The results support a cross-cohort biological axis linking "
            "bulk poor-prognosis biology with a proliferative malignant "
            "state and the opposing direction with a differentiated "
            "hepatic/metabolic malignant state. These labels represent "
            "biological concordance rather than direct clinical-risk "
            "assignment to individual scRNA patients."
        ),

    "Next_step":
        (
            "Step25 will test the robustness and portability of the "
            "MS3 risk-concordant state and the MS3-versus-MS6 axis "
            "across patients before mechanistic downstream analyses."
        ),

    "Step24_annotated_object":
        str(
            STEP24_ANNOTATED_H5AD
        )
}


FINAL_DECISION_PATH = (

    LOGS_DIR /
    "Step24_FINAL_Decision.json"
)


with open(

    FINAL_DECISION_PATH,

    "w",

    encoding="utf-8"

) as f:

    json.dump(

        step24_final_decision,

        f,

        ensure_ascii=False,

        indent=4
    )


print(
    FINAL_DECISION_PATH
)

In [ ]:
# ============================================================
# STEP24 FINAL LOCK - CELL 44
# Final manifest and review
# ============================================================

manifest_rows = []


for category, folder in [

    ("Data", DATA_DIR),
    ("Results", RESULTS_DIR),
    ("Figures", FIGURES_DIR),
    ("Logs", LOGS_DIR)

]:

    if not folder.exists():
        continue


    for path in sorted(
        folder.glob("*")
    ):

        if path.is_file():

            manifest_rows.append({

                "Category":
                    category,

                "Filename":
                    path.name,

                "Size_MB":
                    round(
                        path.stat().st_size
                        /
                        1024**2,
                        3
                    )
            })


manifest_df = pd.DataFrame(
    manifest_rows
)


FINAL_MANIFEST_PATH = (

    RESULTS_DIR /
    "Step24_FINAL_Output_Manifest.csv"
)


manifest_df.to_csv(

    FINAL_MANIFEST_PATH,

    index=False,
    encoding="utf-8-sig"
)


print("=" * 125)
print("STEP24 FINAL REVIEW")
print("=" * 125)


print("\n[1] Final bulk-risk state roles")
print("-" * 125)

display(

    risk_role_table[
        [
            "Final_state_id",
            "Final_state_name",
            "Step24_role",
            "N_eligible_patients",
            "Convergence_mean",
            "Convergence_median",
            "Positive_fraction"
        ]
    ]
    .round(3)
)


print("\n[2] Primary MS3 versus MS6 axis")
print("-" * 125)

display(
    axis_table
    .round(3)
)


print("\n[3] Frozen conclusions")
print("-" * 125)

print(
    "Primary bulk-risk-concordant state: MS3 Cycling-G2M"
)

print(
    "Secondary contextual concordant state: MS1 Hypoxia-TGFb-Inflammatory"
)

print(
    "Supportive concordant state: MS5 Stress-Inflammatory-Progenitor"
)

print(
    "Primary bulk-risk-opposing state: MS6 Hepatic-Differentiated"
)

print(
    "Primary downstream biological axis: MS3 versus MS6"
)

print(
    "Original RiskScore reconstructed: NO"
)

print(
    "Step23 malignant states modified: NO"
)


print("\n[4] Files")
print("-" * 125)


required_files = [

    RISK_ROLE_PATH,
    AXIS_PATH,
    STEP24_ANNOTATED_H5AD,
    FINAL_DECISION_PATH,
    FINAL_MANIFEST_PATH

]


for path in required_files:

    print(
        "[OK]"
        if path.exists()
        else "[MISSING]",
        path.name
    )


print("\n" + "=" * 125)

print(
    "STEP24 STATUS = FROZEN"
)

print(
    "PRIMARY BULK-RISK-CONCORDANT MALIGNANT STATE = MS3"
)

print(
    "PRIMARY BULK-RISK-OPPOSING MALIGNANT STATE = MS6"
)

print(
    "PRIMARY DOWNSTREAM AXIS = MS3 vs MS6"
)

print(
    "NEXT = STEP25 CROSS-PATIENT ROBUSTNESS AND PORTABILITY"
)

print("=" * 125)